DEMO


In [ ]:
import numpy as np
import pandas as pd
import sys
import os

# __ ensure local modules are importable ______________________________________
# sys.path.insert(0, os.path.dirname(__file__)) # This line caused the error

# The following imports are not needed as functions/variables are globally available after cell execution
# from data_loader       import load_and_preprocess, CAT_COLS, NUM_COLS
# from knn_model         import train_knn, evaluate_knn, style_similarity_score
# from naive_bayes_model import train_naive_bayes, predict_occasion

CSV_PATH = "closet_fashion_dataset.csv"

BANNER = """
╔══════════════════════════════════════════════════════╗
║        THE CLOSET FASHION ASSISTANT — DEMO           ║
║        AI-Based Smart Outfit Recommendation          ║
╚══════════════════════════════════════════════════════╝
"""

# These are defined here for use in run_demo's internal logic, even if globally available from data_loader
CAT_COLS = [
    'category', 'color', 'pattern', 'material',
    'occasion_type', 'season_suitability', 'weather_condition'
]
NUM_COLS = [
    'temperature_C', 'wear_count_last30days', 'days_since_last_worn',
    'color_harmony_score', 'occasion_match_score',
    'weather_match_score', 'diversity_score', 'utility_score'
]

def compute_utility(weather_score, occasion_score, style_score, diversity_score):
    return (0.35 * weather_score +
            0.30 * occasion_score +
            0.20 * style_score   +
            0.15 * diversity_score)


def encode_input(user_input: dict, reference_df: pd.DataFrame) -> np.ndarray:
    """
    One-hot encodes a single user input dict to match the training feature space.
    """
    row = pd.DataFrame([user_input])
    # Combine with reference to ensure all columns present after get_dummies
    combined = pd.concat([reference_df[CAT_COLS + NUM_COLS], row], ignore_index=True)
    encoded  = pd.get_dummies(combined, columns=CAT_COLS)
    # Take only the last row (the user input)
    encoded_row = encoded.iloc[[-1]].drop(columns=[c for c in encoded.columns if c not in
                                                    [c for c in encoded.columns]])
    return encoded_row.values.astype(float)

def calculate_contextual_scores(outfit_props: dict, user_weather: str, user_temp: float, user_occasion: str) -> tuple[float, float]:
    """
    Calculates how well an outfit matches the user's current weather and occasion context.
    Returns scores between 0 and 1.
    """
    # Occasion Score
    occasion_score = 0.0
    if outfit_props['occasion_type'].lower() == user_occasion.lower():
        occasion_score = 1.0 # Perfect match
    elif (outfit_props['occasion_type'].lower() in ['casual', 'outdoor'] and user_occasion.lower() in ['casual', 'outdoor']) or \
         (outfit_props['occasion_type'].lower() in ['formal', 'interview', 'wedding'] and user_occasion.lower() in ['formal', 'interview', 'wedding']):
        occasion_score = 0.7 # Good partial match for similar types
    else:
        occasion_score = 0.2 # Low match for distinct occasions

    # Weather Score
    weather_score = 0.0

    # 1. Match based on weather condition (e.g., sunny, rainy, cold)
    if outfit_props['weather_condition'].lower() == user_weather.lower():
        weather_score += 0.4
    else:
        # Penalize strong mismatches (e.g., cold outfit in hot weather)
        if (outfit_props['weather_condition'].lower() == 'cold' and user_weather.lower() in ['sunny', 'hot']) or \
           (outfit_props['weather_condition'].lower() in ['sunny', 'hot'] and user_weather.lower() == 'cold'):
           weather_score += 0.0 # Very bad match
        else:
           weather_score += 0.2 # Slight mismatch

    # 2. Match based on season suitability and temperature range
    if outfit_props['season_suitability'].lower() == 'both':
        weather_score += 0.2 # Versatile outfit gets a boost
    elif outfit_props['season_suitability'].lower() == 'summer' and user_temp >= 20.0: # Arbitrary threshold for summer
        weather_score += 0.2
    elif outfit_props['season_suitability'].lower() == 'winter' and user_temp <= 15.0: # Arbitrary threshold for winter
        weather_score += 0.2

    # 3. Match based on temperature proximity
    temp_diff = abs(outfit_props['temperature_C'] - user_temp)
    if temp_diff <= 5: # Ideal temperature match
        weather_score += 0.4
    elif temp_diff <= 10: # Acceptable temperature match
        weather_score += 0.2

    # Clamp weather_score to [0, 1]
    weather_score = min(1.0, max(0.0, weather_score))

    return occasion_score, weather_score

# NUM_FEATURES is imported from naive_bayes_model.py, but it's not actually imported, so copy here
# This is a temporary fix for now.
NB_NUM_FEATURES = [
    'temperature_C', 'wear_count_last30days', 'days_since_last_worn',
    'color_harmony_score', 'occasion_match_score',
    'weather_match_score', 'diversity_score', 'utility_score'
]

def run_demo(knn_model, nb_model, le, scaler, reference_df, feature_names_for_knn, df_raw_for_sampling):
    """
    Runs the interactive outfit recommendation demo.
    """
    print(BANNER)

    # __ Dynamically generate candidate outfits _____________________________________
    num_candidates = 3
    sampled_outfits = df_raw_for_sampling.sample(n=num_candidates, random_state=42).to_dict(orient='records')

    candidates = []
    for i, outfit_row in enumerate(sampled_outfits):
        # Create a descriptive name for the outfit
        outfit_name = f"Outfit {chr(65+i)} — {outfit_row['color'].capitalize()} {outfit_row['material'].capitalize()} {outfit_row['category'].capitalize()} ({outfit_row['pattern'].capitalize()}, {outfit_row['occasion_type'].capitalize()})"

        # Construct the outfit dictionary with all necessary properties
        candidate_dict = {
            "name": outfit_name,
            "category": outfit_row['category'],
            "color": outfit_row['color'],
            "pattern": outfit_row['pattern'],
            "material": outfit_row['material'],
            "occasion_type": outfit_row['occasion_type'],
            "season_suitability": outfit_row['season_suitability'],
            "weather_condition": outfit_row['weather_condition'],
            "temperature_C": outfit_row['temperature_C'],
            "wear_count_last30days": outfit_row['wear_count_last30days'],
            "days_since_last_worn": outfit_row['days_since_last_worn'],
            "color_harmony_score": outfit_row['color_harmony_score'],
            "occasion_match_score": outfit_row['occasion_match_score'],
            "weather_match_score": outfit_row['weather_match_score'],
            "diversity_score": outfit_row['diversity_score'],
            "utility_score": outfit_row['utility_score']
        }
        candidates.append(candidate_dict)


    # __ Get user input for current context ________________________________
    print("\nPlease tell me about the current situation:")
    user_weather = input("  What is the current weather condition (e.g., sunny, rainy, cold): ").strip().lower()
    while True:
        try:
            user_temp_str = input("  What is the current temperature in Celsius (e.g., 25.0): ").strip()
            user_temp = float(user_temp_str)
            break
        except ValueError:
            print("  Invalid temperature. Please enter a number.")
    user_occasion = input("  What is the desired occasion (e.g., casual, formal, party): ").strip().lower()
    print()

    print("  Current Context:")
    print("  ─────────────────────────────────────────────")
    print(f"  📍 Weather   : {user_weather.capitalize()}, {user_temp}°C")
    print(f"  👔 Occasion  : {user_occasion.capitalize()} outing")
    print(f"  🧺 Wardrobe  : {num_candidates} candidate outfits randomly selected")
    print("  ─────────────────────────────────────────────\n")

    results = []
    print("  Evaluating candidates...\n")

    for i, outfit_data in enumerate(candidates): # Renamed 'outfit' to 'outfit_data' to avoid confusion
        outfit = outfit_data.copy() # Create a copy to avoid modifying the original dictionary
        name = outfit.pop('name') # Extract 'name' and remove it from 'outfit'

        # Dynamically calculate contextual scores based on user input
        contextual_occasion_score, contextual_weather_score = calculate_contextual_scores(
            outfit, user_weather, user_temp, user_occasion
        )

        # __ KNN: predict acceptance ______________________________________
        outfit_df = pd.DataFrame([outfit]) # Create a DataFrame from the outfit dict
        outfit_df_encoded = pd.get_dummies(outfit_df, columns=CAT_COLS)

        # Reindex to ensure all feature_names_for_knn are present, filling missing with 0
        outfit_aligned = outfit_df_encoded.reindex(columns=feature_names_for_knn, fill_value=0)

        outfit_vec = outfit_aligned.values.astype(float) # Converts to numpy array
        outfit_vec_norm = scaler.transform(outfit_vec)[0]

        try:
            knn_pred  = knn_model.predict([outfit_vec_norm[:knn_model.n_features_in_]])[0]
            knn_proba = knn_model.predict_proba([outfit_vec_norm[:knn_model.n_features_in_]])[0][1]
        except Exception:
            # Fallback if prediction fails (e.g., due to mismatch in features)
            knn_pred, knn_proba = 1, outfit.get('utility_score', 0.5) # Use existing utility_score as a proxy

        # __ NB: predict occasion _________________________________________
        nb_features = {
            'temperature_C'          : user_temp, # Use user-provided temperature for NB prediction
            'wear_count_last30days'  : outfit['wear_count_last30days'],
            'days_since_last_worn'   : outfit['days_since_last_worn'],
            'color_harmony_score'    : outfit['color_harmony_score'],
            'occasion_match_score'   : contextual_occasion_score, # Use contextual score for NB
            'weather_match_score'    : contextual_weather_score,  # Use contextual score for NB
            'diversity_score'        : outfit['diversity_score'],
            'utility_score'          : outfit['utility_score'], # Use outfit's inherent utility_score for NB input feature
        }
        # Create a DataFrame with the correct feature names for NB and convert to values
        nb_input_df = pd.DataFrame([nb_features], columns=NB_NUM_FEATURES)
        nb_pred_idx = nb_model.predict(nb_input_df.values)[0]
        nb_occasion = le.inverse_transform([nb_pred_idx])[0]

        # __ Utility score _________________________________________________
        utility = compute_utility(
            contextual_weather_score,
            contextual_occasion_score,
            knn_proba,
            outfit['diversity_score']
        )

        results.append({
            "name"      : name,
            "knn_pred"  : knn_pred,
            "knn_proba" : knn_proba,
            "nb_occasion": nb_occasion,
            "utility"   : utility,
            "outfit"    : outfit
        })

        status = "✅ ACCEPTED" if knn_pred == 1 else "❌ REJECTED"
        print(f"  [{i+1}] {name}")
        print(f"       KNN Prediction    : {status}  (confidence: {knn_proba:.2f})")
        print(f"       NB Occasion Match : {nb_occasion.upper()}")
        print(f"       Utility Score     : {utility:.4f}")
        print()

    # __ Select best outfit (highest utility) _____________________________
    accepted = [r for r in results if r['knn_pred'] == 1]
    pool = accepted if accepted else results
    best = max(pool, key=lambda r: r['utility'])

    print("  ╔══════════════════════════════════════════════╗")
    print("  ║         🏆  RECOMMENDED OUTFIT               ║")
    print("  ╠══════════════════════════════════════════════╣")
    print(f"  ║  {best['name'][:46]:<46}   ║")
    print(f"  ║  Utility Score : {best['utility']:.4f}                         ║")
    print(f"  ║  Occasion Match: {best['nb_occasion'].upper():<28}  ║")
    print("  ╚══════════════════════════════════════════════╝")
    print()
    print("  [Agent Action] Outfit preview displayed to user.")
    print("  [Agent Action] Wear history updated.")
    print("  [Agent Action] Item marked as 'worn today'.\n")


def main():
    print("\nLoading dataset and training models...\n")

    # Train KNN
    X_train, X_test, y_train, y_test, features, scaler = \
        load_and_preprocess(CSV_PATH)
    knn = train_knn(X_train, y_train, k=5)
    metrics = evaluate_knn(knn, X_test, y_test)

    # Train Logistic Regression for occasion prediction
    # Replacing Naive Bayes with Logistic Regression due to poor NB performance
    nb, le, nb_acc = train_logistic_regression(CSV_PATH)

    # Load raw df for encoding reference
    df_raw = pd.read_csv(CSV_PATH)

    # Run demo
    run_demo(knn, nb, le, scaler, df_raw, features, df_raw)

    print("\n  Model Summary:")
    print(f"  KNN  Accuracy : {metrics['accuracy']*100:.1f}%")
    print(f"  KNN  F1-Score : {metrics['f1']:.4f}")
    print(f"  Occasion Classifier Accuracy : {nb_acc*100:.1f}%") # Updated label
    print("\n  Demo complete. ✓")


if __name__ == "__main__":
    main()


Loading dataset and training models...

[data_loader] Dataset loaded: 500 records, 48 features
[data_loader] Train: 400 | Test: 100
[data_loader] Class balance — Accepted: 289 | Rejected: 211
[knn_model] KNN model trained with k=5

        KNN MODEL — EVALUATION RESULTS
  Accuracy  : 0.5100  (51.0%)
  Precision : 0.5672
  Recall    : 0.6552
  F1-Score  : 0.6080

  Confusion Matrix:
             Predicted 0   Predicted 1
  Actual 0 :       13            29
  Actual 1 :       20            38

  Full Classification Report:
              precision    recall  f1-score   support

    Rejected       0.39      0.31      0.35        42
    Accepted       0.57      0.66      0.61        58

    accuracy                           0.51       100
   macro avg       0.48      0.48      0.48       100
weighted avg       0.49      0.51      0.50       100


    LOGISTIC REGRESSION — OCCASION CLASSIFIER RESULTS
  Accuracy        : 0.1500  (15.0%) felt
  CV Accuracy     : 0.1460 ± 0.0307
  Classes    

In [ ]:
!pip install PyPDF2
import PyPDF2

def read_pdf(file_path):
    with open(file_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ''
        for page in reader.pages:
            text += page.extract_text()
    return text

plan_content = read_pdf('/content/ML_Component_Plan.pdf')
print(plan_content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 2.0 MB/s eta 0:00:00
1 
 Riphah International University Gulberg Green Campus 
Islamabad  
Faculty of Computing  
 
 
Artificial Intelligence  
 
Project Report  
  
The Closet Fashion Assistant  
 (AI-Based Smart Outfit Recommendation System)  
 
Submitted By:  
Name  Program  Semester  SAP ID  
Afifa A mna BS-SE 6th 55466  
Anosha Pervaiz Awan  BS-SE 6th 56200  
Areeza Afridi  BS-SE 6th 55254  
Nighat Naseem  BS-SE 6th 53740  
Nimra Tariq  BS-SE 6th  54909  
Simaab Malik  BS-SE 6th  54910  
 
Submitted to: Ma’am Shumaila Qayyu m 
Date of Submission: 09/3/2026  
 
2 
 Table of Contents  
1. Introduction  ................................ ................................ ................................ ................................ ... 3 
2. ML Technique Selection  ................................ ................................ ................................ ..................  3 
2.1 Chosen Technique: K -Nearest Neighbo

DATA LOADER

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split


# ── Column definitions ──────────────────────────────────────────────────────
CAT_COLS = [
    'category', 'color', 'pattern', 'material',
    'occasion_type', 'season_suitability', 'weather_condition'
]
NUM_COLS = [
    'temperature_C', 'wear_count_last30days', 'days_since_last_worn',
    'color_harmony_score', 'occasion_match_score',
    'weather_match_score', 'diversity_score', 'utility_score'
]
# TARGET      = 'outfit_accepted' # Moved inside function
# DROP_COLS   = ['item_id', TARGET] # Moved inside function


def load_and_preprocess(csv_path: str, test_size: float = 0.2, random_state: int = 42):
    """
    Loads the CSV, one-hot encodes categoricals, normalises numerics,
    and returns a stratified 80/20 train/test split.

    Returns
    -------
    X_train, X_test, y_train, y_test : np.ndarray / pd.Series
    feature_names                     : list[str]
    scaler                            : fitted MinMaxScaler
    """
    df = pd.read_csv(csv_path)

    # Define TARGET and DROP_COLS locally to prevent global variable conflicts
    TARGET = 'outfit_accepted'
    DROP_COLS = ['item_id', TARGET]

    # One-hot encode categorical columns
    df_enc = pd.get_dummies(df, columns=CAT_COLS)

    # Separate features and target
    X = df_enc.drop(columns=DROP_COLS)
    y = df_enc[TARGET]
    feature_names = list(X.columns)

    # Normalise all numeric features to [0, 1]
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X.values) # Fit scaler on values, not DataFrame, to avoid warning

    # Stratified split so both classes are proportional in each set
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    print(f"[data_loader] Dataset loaded: {len(df)} records, {len(feature_names)} features")
    print(f"[data_loader] Train: {len(X_train)} | Test: {len(X_test)}")
    print(f"[data_loader] Class balance — Accepted: {y.sum()} | Rejected: {(y==0).sum()}")

    return X_train, X_test, y_train, y_test, feature_names, scaler


if __name__ == "__main__":
    load_and_preprocess("closet_fashion_dataset.csv")

[data_loader] Dataset loaded: 500 records, 48 features
[data_loader] Train: 400 | Test: 100
[data_loader] Class balance — Accepted: 289 | Rejected: 211


KNN MODEL

In [ ]:
"""
knn_model.py
============
The Closet Fashion Assistant — ML Component
KNN classifier: predicts outfit acceptance & computes style similarity.
"""

import numpy as np
import pickle
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)


def find_best_k(X_train: np.ndarray, y_train, k_range=range(1, 16, 2)) -> int:
    """
    Runs 5-fold cross-validation over odd k values and returns the best k.
    """
    print("\n[knn_model] Finding best k via 5-fold cross-validation...")
    best_k, best_score = 1, 0
    for k in k_range:
        knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
        scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='f1')
        mean_score = scores.mean()
        print(f"  k={k:2d}  →  CV F1 = {mean_score:.4f} \u00b1 {scores.std():.4f}")
        if mean_score > best_score:
            best_score, best_k = mean_score, k
    print(f"[knn_model] Best k = {best_k}  (CV F1 = {best_score:.4f})")
    return best_k


def train_knn(X_train: np.ndarray, y_train, k: int = None) -> KNeighborsClassifier:
    """
    Trains a KNN classifier. Finds best k automatically if not provided.
    """
    if k is None:
        k = find_best_k(X_train, y_train)
    knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean', algorithm='ball_tree')
    knn.fit(X_train, y_train)
    print(f"[knn_model] KNN model trained with k={k}")
    return knn


def evaluate_knn(model: KNeighborsClassifier, X_test: np.ndarray, y_test) -> dict:
    """
    Evaluates the model and prints a full performance report.
    Returns a dict of all metrics.
    """
    y_pred = model.predict(X_test)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    cm   = confusion_matrix(y_test, y_pred)

    print("\n" + "="*55)
    print("        KNN MODEL \u2014 EVALUATION RESULTS")
    print("="*55)
    print(f"  Accuracy  : {acc:.4f}  ({acc*100:.1f}%)")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print("\n  Confusion Matrix:")
    print(f"             Predicted 0   Predicted 1")
    print(f"  Actual 0 :   {cm[0][0]:6d}        {cm[0][1]:6d}")
    print(f"  Actual 1 :   {cm[1][0]:6d}        {cm[1][1]:6d}")
    print("\n  Full Classification Report:")
    print(classification_report(y_test, y_pred, target_names=["Rejected", "Accepted"]))
    print("="*55)

    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1, "confusion_matrix": cm}


def style_similarity_score(model: KNeighborsClassifier, outfit_vector: np.ndarray) -> float:
    """
    Computes a style similarity score (0\u20131) for a single outfit vector.
    Uses the KNN distance to the k nearest neighbours \u2014 closer = more similar.
    """
    distances, _ = model.kneighbors([outfit_vector])
    avg_dist = distances[0].mean()
    # Convert distance to similarity: closer neighbours \u2192 higher score
    score = 1 / (1 + avg_dist)
    return round(float(score), 4)


def save_model(model, path: str = "knn_model.pkl"):
    with open(path, 'wb') as f:
        pickle.dump(model, f)
    print(f"[knn_model] Model saved to {path}")


def load_model(path: str = "knn_model.pkl") -> KNeighborsClassifier:
    with open(path, 'rb') as f:
        return pickle.load(f)


if __name__ == "__main__":
    # Removed: from data_loader import load_and_preprocess
    X_train, X_test, y_train, y_test, _, _ = load_and_preprocess("closet_fashion_dataset.csv")
    model = train_knn(X_train, y_train)
    evaluate_knn(model, X_test, y_test)
    save_model(model)

NAIVE BAYES MODEL

In [ ]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import pickle


NUM_FEATURES = [
    'temperature_C', 'wear_count_last30days', 'days_since_last_worn',
    'color_harmony_score', 'occasion_match_score',
    'weather_match_score', 'diversity_score', 'utility_score'
]
TARGET = 'occasion_type'


def train_naive_bayes(csv_path: str):
    """
    Trains a Gaussian Naive Bayes classifier to predict occasion_type.
    Returns the trained model, label encoder, and test metrics.
    """
    df = pd.read_csv(csv_path)

    X = df[NUM_FEATURES]
    le = LabelEncoder()
    y = le.fit_transform(df[TARGET])   # encode string labels → integers

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    nb = GaussianNB()
    nb.fit(X_train.values, y_train) # Fit on values to avoid warning
    y_pred = nb.predict(X_test.values)

    acc = accuracy_score(y_test, y_pred)
    cv_scores = cross_val_score(nb, X.values, y, cv=5, scoring='accuracy')

    print("\n" + "="*55)
    print("    NAIVE BAYES — OCCASION CLASSIFIER RESULTS")
    print("="*55)
    print(f"  Accuracy        : {acc:.4f}  ({acc*100:.1f}%) felt")
    print(f"  CV Accuracy     : {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}")
    print(f"  Classes         : {list(le.classes_)}")
    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    print("="*55)

    return nb, le, acc


def predict_occasion(model: GaussianNB, le: LabelEncoder, features: dict) -> str:
    """
    Predicts the most suitable occasion for a given outfit.
    features: dict with keys matching NUM_FEATURES
    """
    x = np.array([[features[f] for f in NUM_FEATURES]])
    pred = model.predict(x)[0]
    proba = model.predict_proba(x)[0]
    occasion = le.inverse_transform([pred])[0]

    print(f"\n[NaiveBayes] Predicted occasion: {occasion.upper()}")
    print(f"[NaiveBayes] Probabilities:")
    for cls, p in sorted(zip(le.classes_, proba), key=lambda x: -x[1]):
        bar = "\u2588" * int(p * 20)
        print(f"   {cls:12s}: {p:.3f}  {bar}")

    return occasion


def save_nb_model(model, le, model_path="nb_model.pkl", le_path="nb_le.pkl"):
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    with open(le_path, 'wb') as f:
        pickle.dump(le, f)
    print(f"[naive_bayes] Models saved.")


if __name__ == "__main__":
    nb, le, acc = train_naive_bayes("closet_fashion_dataset.csv")
    save_nb_model(nb, le)

    # Quick demo prediction
    sample = {
        'temperature_C': 32.0,
        'wear_count_last30days': 1,
        'days_since_last_worn': 20,
        'color_harmony_score': 0.85,
        'occasion_match_score': 0.90,
        'weather_match_score': 0.88,
        'diversity_score': 0.93,
        'utility_score': 0.88
    }
    predict_occasion(nb, le, sample)


    NAIVE BAYES — OCCASION CLASSIFIER RESULTS
  Accuracy        : 0.2000  (20.0%) felt
  CV Accuracy     : 0.1660 ± 0.0233
  Classes         : ['casual', 'formal', 'interview', 'outdoor', 'party', 'wedding']

  Classification Report:
              precision    recall  f1-score   support

      casual       0.38      0.35      0.36        17
      formal       0.09      0.06      0.07        17
   interview       0.00      0.00      0.00        14
     outdoor       0.17      0.14      0.15        14
       party       0.16      0.26      0.20        19
     wedding       0.24      0.32      0.27        19

    accuracy                           0.20       100
   macro avg       0.17      0.19      0.18       100
weighted avg       0.18      0.20      0.18       100

[naive_bayes] Models saved.

[NaiveBayes] Predicted occasion: WEDDING
[NaiveBayes] Probabilities:
   wedding     : 0.421  ████████
   casual      : 0.269  █████
   party       : 0.122  ██
   formal      : 0.079  █
   outdo

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

def train_logistic_regression(csv_path: str):
    """
    Trains a Logistic Regression classifier to predict occasion_type.
    Returns the trained model, label encoder, and test metrics.
    """
    df = pd.read_csv(csv_path)

    X = df[NUM_FEATURES]
    le = LabelEncoder()
    y = le.fit_transform(df[TARGET])   # encode string labels → integers

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    lr_model = LogisticRegression(max_iter=1000, solver='liblinear') # Increased max_iter for convergence
    lr_model.fit(X_train.values, y_train) # Fit on values to avoid warning
    y_pred = lr_model.predict(X_test.values)

    acc = accuracy_score(y_test, y_pred)
    cv_scores = cross_val_score(lr_model, X.values, y, cv=5, scoring='accuracy')

    print("\n" + "="*55)
    print("    LOGISTIC REGRESSION — OCCASION CLASSIFIER RESULTS")
    print("="*55)
    print(f"  Accuracy        : {acc:.4f}  ({acc*100:.1f}%) felt")
    print(f"  CV Accuracy     : {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}")
    print(f"  Classes         : {list(le.classes_)}")
    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    print("="*55)

    return lr_model, le, acc

def train_decision_tree(csv_path: str):
    """
    Trains a Decision Tree Classifier to predict occasion_type.
    Returns the trained model, label encoder, and test metrics.
    """
    df = pd.read_csv(csv_path)

    X = df[NUM_FEATURES]
    le = LabelEncoder()
    y = le.fit_transform(df[TARGET])   # encode string labels → integers

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    dt_model = DecisionTreeClassifier(random_state=42)
    dt_model.fit(X_train.values, y_train)
    y_pred = dt_model.predict(X_test.values)

    acc = accuracy_score(y_test, y_pred)
    cv_scores = cross_val_score(dt_model, X.values, y, cv=5, scoring='accuracy')

    print("\n" + "="*55)
    print("    DECISION TREE — OCCASION CLASSIFIER RESULTS")
    print("="*55)
    print(f"  Accuracy        : {acc:.4f}  ({acc*100:.1f}%) felt")
    print(f"  CV Accuracy     : {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}")
    print(f"  Classes         : {list(le.classes_)}")
    print("\n  Classification Report:")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    print("="*55)

    return dt_model, le, acc

In [ ]:
# Update the main function to use the new Decision Tree model
def main():
    print("\nLoading dataset and training models...\n")

    # Train KNN
    X_train, X_test, y_train, y_test, features, scaler = \
        load_and_preprocess(CSV_PATH)
    knn = train_knn(X_train, y_train, k=5)
    metrics = evaluate_knn(knn, X_test, y_test)

    # Train Decision Tree for occasion prediction (replacing Logistic Regression)
    nb, le, nb_acc = train_decision_tree(CSV_PATH)

    # Load raw df for encoding reference
    df_raw = pd.read_csv(CSV_PATH)

    # Run demo
    run_demo(knn, nb, le, scaler, df_raw, features, df_raw)

    print("\n  Model Summary:")
    print(f"  KNN  Accuracy : {metrics['accuracy']*100:.1f}%")
    print(f"  KNN  F1-Score : {metrics['f1']:.4f}")
    print(f"  Occasion Classifier Accuracy : {nb_acc*100:.1f}%") # Updated label
    print("\n  Demo complete. \u2713")


if __name__ == "__main__":
    main()


Loading dataset and training models...

[data_loader] Dataset loaded: 500 records, 48 features
[data_loader] Train: 400 | Test: 100
[data_loader] Class balance — Accepted: 289 | Rejected: 211
[knn_model] KNN model trained with k=5

        KNN MODEL — EVALUATION RESULTS
  Accuracy  : 0.5100  (51.0%)
  Precision : 0.5672
  Recall    : 0.6552
  F1-Score  : 0.6080

  Confusion Matrix:
             Predicted 0   Predicted 1
  Actual 0 :       13            29
  Actual 1 :       20            38

  Full Classification Report:
              precision    recall  f1-score   support

    Rejected       0.39      0.31      0.35        42
    Accepted       0.57      0.66      0.61        58

    accuracy                           0.51       100
   macro avg       0.48      0.48      0.48       100
weighted avg       0.49      0.51      0.50       100


    DECISION TREE — OCCASION CLASSIFIER RESULTS
  Accuracy        : 0.1400  (14.0%) felt
  CV Accuracy     : 0.1460 ± 0.0326
  Classes         :

In [ ]:
import pandas as pd

def analyze_feature_importance(model, feature_names):
    print("\n=======================================================")
    print("        DECISION TREE FEATURE IMPORTANCE")
    print("=======================================================")
    if hasattr(model, 'feature_importances_'):
        importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': model.feature_importances_
        }).sort_values(by='Importance', ascending=False)
        print(importance_df.to_string(index=False))
    else:
        print("Model does not have feature_importances_ attribute.")
    print("=======================================================")

# Re-run main to get the trained Decision Tree model (nb) and its feature names (NUM_FEATURES)
# Note: This will re-train the models and re-run the demo, but it's necessary to access the latest trained model instance.
# For a more direct approach without re-running the full demo, we could save/load the model.

print("\nRe-running main function to capture trained Decision Tree model for feature importance analysis...")

# Temporarily redirect stdout to capture main's output without re-displaying it
import io
import sys

old_stdout = sys.stdout
sys.stdout = new_stdout = io.StringIO()

main() # This re-executes the demo

sys.stdout = old_stdout # Restore stdout

# Extract the nb_model and NUM_FEATURES from the global scope after main() has run
# Assuming main() populates `nb` and `NUM_FEATURES` in the global scope
# If not directly accessible, might need modifications in main() to return them or store them.
# For now, let's assume `nb` (the trained Decision Tree) and `NUM_FEATURES` are available globally after main.

# The Decision Tree model is returned as 'nb' from train_decision_tree in main()
# The NUM_FEATURES list is defined globally in the earlier cells.

# If nb_model and NUM_FEATURES are not directly accessible, the following would need adjustment
# based on how they are exposed after `main()` runs in the actual environment.
# For this controlled environment, we can assume they are available.

# The nb_model is the Decision Tree model trained in main(), and NUM_FEATURES is defined in a previous cell.
# Let's make sure to use the correct variables that are populated by the `main()` call.
# Based on the notebook state, `nb` will be the Decision Tree model and `NUM_FEATURES` is a global list.

analyze_feature_importance(nb, NUM_FEATURES)



Re-running main function to capture trained Decision Tree model for feature importance analysis...


EVALUATOR

In [ ]:
"""
evaluator.py
============
The Closet Fashion Assistant — ML Component
Generates performance analysis: accuracy, F1, confusion matrix,
k-value curve, and complexity analysis. Saves plots as PNG files.
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')   # non-interactive backend (no display needed)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import os

OUTPUT_DIR = "performance_plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)


def plot_k_vs_accuracy(X_train, y_train, k_range=range(1, 20, 2)):
    """Plots CV accuracy for different k values to justify k=5."""
    accs, stds = [], []
    for k in k_range:
        knn = KNeighborsClassifier(n_neighbors=k, metric='euclidean')
        scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
        accs.append(scores.mean())
        stds.append(scores.std())

    accs, stds = np.array(accs), np.array(stds)
    ks = list(k_range)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(ks, accs, 'o-', color='#2E5FA3', linewidth=2, markersize=7, label='Mean CV Accuracy')
    ax.fill_between(ks, accs - stds, accs + stds, alpha=0.2, color='#2E5FA3', label='\u00b1 1 Std Dev')
    best_k = ks[np.argmax(accs)]
    ax.axvline(best_k, color='red', linestyle='--', linewidth=1.5, label=f'Best k = {best_k}')
    ax.set_xlabel('k (Number of Neighbors)', fontsize=12)
    ax.set_ylabel('CV Accuracy', fontsize=12)
    ax.set_title('KNN: k-Value vs. Cross-Validation Accuracy', fontsize=13, fontweight='bold')
    ax.legend()
    ax.set_xticks(ks)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = f"{OUTPUT_DIR}/k_vs_accuracy.png"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"[evaluator] Saved: {path}")
    return path


def plot_confusion_matrix(y_test, y_pred, title="KNN Confusion Matrix", filename="knn_confusion_matrix.png"):
    """Plots and saves confusion matrix."""
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Rejected", "Accepted"])
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    path = f"{OUTPUT_DIR}/{filename}"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"[evaluator] Saved: {path}")
    return path


def plot_metrics_bar(metrics_knn: dict, metrics_nb: dict = None):
    """Bar chart comparing KNN (and NB if provided) on key metrics."""
    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    knn_vals = [metrics_knn['accuracy'], metrics_knn['precision'],
                metrics_knn['recall'], metrics_knn['f1']]

    x = np.arange(len(metric_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 4))
    bars1 = ax.bar(x - width/2, knn_vals, width, label='KNN', color='#2E5FA3', alpha=0.85)

    if metrics_nb:
        nb_vals = [metrics_nb['accuracy'], metrics_nb.get('precision', 0),
                   metrics_nb.get('recall', 0), metrics_nb.get('f1', 0)]
        bars2 = ax.bar(x + width/2, nb_vals, width, label='Naive Bayes', color='#E8943A', alpha=0.85)

    # Value labels on bars
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)

    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metric_names)
    ax.set_ylim(0, 1.15)
    ax.legend()
    ax.axhline(0.80, color='green', linestyle='--', linewidth=1, alpha=0.5, label='Target \u2265 0.80')
    ax.grid(True, alpha=0.2, axis='y')
    plt.tight_layout()
    path = f"{OUTPUT_DIR}/metrics_comparison.png"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"[evaluator] Saved: {path}")
    return path


def plot_complexity():
    """Illustrates KNN time complexity O(n*d) vs dataset size."""
    sizes = np.arange(100, 5001, 100)
    d = 40  # approximate feature count after encoding
    # Simulated prediction times proportional to O(n*d)
    np.random.seed(42)
    times = (sizes * d / 1e6) + np.random.normal(0, 0.005, len(sizes))
    times = np.abs(times)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(sizes, times * 1000, color='#2E5FA3', linewidth=2)
    ax.axvline(500, color='red', linestyle='--', linewidth=1.5, label='Current Dataset (n=500)')
    ax.fill_between(sizes, 0, times * 1000, alpha=0.1, color='#2E5FA3')
    ax.set_xlabel('Training Set Size (n)', fontsize=12)
    ax.set_ylabel('Prediction Time (ms)', fontsize=12)
    ax.set_title('KNN Time Complexity: O(n \u00d7 d)', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = f"{OUTPUT_DIR}/complexity_analysis.png"
    fig.savefig(path, dpi=150)
    plt.close()
    print(f"[evaluator] Saved: {path}")
    return path


def print_performance_analysis(metrics: dict):
    """Prints structured strengths/limitations/complexity analysis."""
    print("\n" + "="*55)
    print("         PERFORMANCE ANALYSIS REPORT")
    print("="*55)

    acc = metrics['accuracy']
    f1  = metrics['f1']

    print(f"\n  Accuracy : {acc:.4f} ({'\u2713 MEETS target \u2265 80%' if acc >= 0.80 else '\u2717 Below target'})")
    print(f"  F1-Score : {f1:.4f} ({'\u2713 MEETS target \u2265 0.78' if f1 >= 0.78 else '\u2717 Below target'})")

    print("\n  STRENGTHS:")
    print("  \u2713 No retraining needed \u2014 new liked outfits improve model instantly")
    print("  \u2713 Interpretable \u2014 'you liked 5 similar outfits before'")
    print("  \u2713 Performs well on small, personal datasets (< 1000 items)")
    print("  \u2713 Handles non-linear, multi-feature fashion preferences naturally")
    print("  \u2713 k optimized via cross-validation to prevent overfitting")

    print("\n  LIMITATIONS:")
    print("  \u2717 Cold start problem \u2014 poor predictions for brand new users")
    print("  \u2717 Slow for very large wardrobes (O(n\u00d7d) per prediction)")
    print("  \u2717 Sensitive to irrelevant features \u2014 feature selection matters")
    print("  \u2717 Assumes Euclidean distance is meaningful for fashion features")

    print("\n  COMPLEXITY:")
    print("  \u2022 Training  : O(1) \u2014 KNN stores data, no computation at train time")
    print("  \u2022 Prediction: O(n \u00d7 d) \u2014 compares to all n training items, d features")
    print("  \u2022 Space     : O(n \u00d7 d) \u2014 stores entire training dataset")
    print("  \u2022 For n=500, d=40 \u2192 ~20,000 operations per prediction")
    print("  \u2022 Target    : < 2 seconds on mobile device \u2713")
    print("="*55)


if __name__ == "__main__":
    # from data_loader import load_and_preprocess
    # from knn_model import train_knn, evaluate_knn

    X_train, X_test, y_train, y_test, _, _ = load_and_preprocess("closet_fashion_dataset.csv")
    model = train_knn(X_train, y_train, k=5)
    metrics = evaluate_knn(model, X_test, y_test)

    print_performance_analysis(metrics)
    plot_k_vs_accuracy(X_train, y_train)
    y_pred = model.predict(X_test)
    plot_confusion_matrix(y_test, y_pred)
    plot_metrics_bar(metrics)
    plot_complexity()
    print("\n[evaluator] All plots saved to ./performance_plots/")